In [ ]:
import csv
import re
import urllib
from time import sleep
import requests

In [ ]:
import urllib.request
import urllib.parse
import json

query= "keratan sulfate I"

# --------- PASSO 1: SEARCH ---------
base_search = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": query,
    "retmax": 5,              # número de artigos
    "retmode": "json"
}

search_url = base_search + "?" + urllib.parse.urlencode(params)

with urllib.request.urlopen(search_url) as f:
    search_data = json.loads(f.read().decode("utf-8"))

id_list = search_data["esearchresult"]["idlist"]

print("IDs encontrados:", id_list)


# --------- PASSO 2: FETCH (ABSTRACTS) ---------
base_fetch = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

fetch_params = {
    "db": "pubmed",
    "id": ",".join(id_list),
    "rettype": "abstract",
    "retmode": "text"
}

fetch_url = base_fetch + "?" + urllib.parse.urlencode(fetch_params)

with urllib.request.urlopen(fetch_url) as f:
    abstracts = f.read().decode("utf-8")

print("\n--- ABSTRACTS ---\n")
print(abstracts)

IDs encontrados: ['41616943', '41614872', '41513091', '41499318', '41475749']

--- ABSTRACTS ---

1. Transplant Cell Ther. 2026 Jan 28:S2666-6367(26)00063-1. doi: 
10.1016/j.jtct.2026.01.029. Online ahead of print.

Allogeneic Hematopoietic Cell Transplantation for Morquio A Syndrome: An 
International Retrospective Study.

Kharbanda S(1), Cho E(2), Chen J(3), Zhang M(3), Yalcin K(4), Yesilipek A(4), 
Yabe H(5), Lindemans C(6), van Hasselt P(6), Alghasi A(7), Fong N(8), 
Bhattacharya R(9), Sheikh I(10), Guilcher GMT(11), Grimard P(11), Tomatsu S(12), 
Dvorak CC(2).

Author information:
(1)Division of Pediatric Allergy, Immunology, and Bone Marrow Transplant Benioff 
Children's Hospital, University of California San Francisco, San Francisco, 
California. Electronic address: sandhya.kharbanda@ucsf.edu.
(2)Division of Pediatric Allergy, Immunology, and Bone Marrow Transplant Benioff 
Children's Hospital, University of California San Francisco, San Francisco, 
California.
(3)Department of 

In [13]:
fetch_url

'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id=41616943%2C41614872%2C41513091%2C41499318%2C41475749&rettype=abstract&retmode=text'

In [14]:
search_url

'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=keratan+sulfate+I&retmax=5&retmode=json'

In [ ]:
import urllib.request
import urllib.parse
import json
import xml.etree.ElementTree as ET
import pandas as pd
import os

# A tua entidade do modelo (Camada 1)
entity_id_no_modelo = "M_m02278s__91__s__93__" 
query = "keratan sulfate I"

# Estruturas para guardar os dados
publications_data = []
edges_data = []

# --------- PASSO 1: SEARCH ---------
print(f"🔍 A procurar artigos para: {query}")
base_search = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {"db": "pubmed", "term": query, "retmax": 5, "retmode": "json"}
search_url = base_search + "?" + urllib.parse.urlencode(params)

with urllib.request.urlopen(search_url) as f:
    search_data = json.loads(f.read().decode("utf-8"))
id_list = search_data.get("esearchresult", {}).get("idlist", [])

if id_list:
    print(f"✅ {len(id_list)} IDs encontrados: {id_list}")

    # --------- PASSO 2: FETCH (XML para dados estruturados) ---------
    # Usamos XML porque é a forma mais fiável de extrair Título, Resumo e Autores separados
    base_fetch = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {"db": "pubmed", "id": ",".join(id_list), "retmode": "xml"}
    fetch_url = base_fetch + "?" + urllib.parse.urlencode(fetch_params)
    
    with urllib.request.urlopen(fetch_url) as f:
        xml_data = f.read()
        
    root = ET.fromstring(xml_data)
    
    for article in root.findall('.//PubmedArticle'):
        pmid = article.find('.//PMID').text
        
        # Extrair o Título
        title_node = article.find('.//ArticleTitle')
        title = title_node.text if title_node is not None else ""
        
        # Extrair o Abstract
        abstract_node = article.find('.//AbstractText')
        abstract = abstract_node.text if abstract_node is not None else ""
        
        # Extrair o Ano
        year_node = article.find('.//PubDate/Year')
        year = year_node.text if year_node is not None else ""
        
        # Extrair Autores (Apenas o primeiro autor para simplificar, ou juntar todos)
        author_last = article.find('.//AuthorList/Author/LastName')
        authors = author_last.text + " et al." if author_last is not None else ""
        
        journal_node = article.find('.//Journal/Title')
        journal = journal_node.text if journal_node is not None else ""

        doi_node = article.find('.//ArticleId[@IdType="doi"]')
        if doi_node is None:
            doi_node = article.find('.//ELocationID[@EIdType="doi"]')   
        doi = doi_node.text if doi_node is not None else ""
        
        pubmed_url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

        publications_data.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "year": year,
            "authors": authors,
            "journal": journal,
            "doi": doi,                 # O teu novo DOI corrigido
            "pubmed_url": pubmed_url    # O link de fallback perfeito!
        })
        
        # 2. Guardar os dados da Aresta (Entity <-> Publication)
        edges_data.append({
            "entity_id": entity_id_no_modelo,
            "pmid": pmid
        })

# --------- PASSO 3: GERAR OS CSVs ---------
out_dir = "data/pubmed"
os.makedirs(out_dir, exist_ok=True)

# Gravar publications.csv
df_pubs = pd.DataFrame(publications_data)
# Remover duplicados caso o mesmo artigo apareça para várias queries
df_pubs.drop_duplicates(subset=['pmid'], inplace=True) 
df_pubs.to_csv(f"{out_dir}/publications.csv", index=False)
print(f"📄 Guardado: {out_dir}/publications.csv")

# Gravar entity_publication.csv
df_edges = pd.DataFrame(edges_data)
df_edges.to_csv(f"{out_dir}/entity_publication.csv", index=False)
print(f"🔗 Guardado: {out_dir}/entity_publication.csv")

🔍 A procurar artigos para: keratan sulfate I
✅ 5 IDs encontrados: ['41616943', '41614872', '41513091', '41499318', '41475749']
📄 Guardado: data/pubmed/publications.csv
🔗 Guardado: data/pubmed/entity_publication.csv


In [43]:
%pip install neo4j

  Using cached neo4j-6.1.0-py3-none-any.whl.metadata (5.3 kB)
Using cached neo4j-6.1.0-py3-none-any.whl (325 kB)
Note: you may need to restart the kernel to use updated packages.


In [53]:
from neo4j import GraphDatabase

# 1. Ligar ao Neo4j
uri = "bolt://localhost:7687"
driver = GraphDatabase.driver(uri, auth=("neo4j", "password")) # Põe a tua password

# =================================================================
# PASSO 1: Achar todos os IDs verdadeiros no Grafo usando o Nome Base
# =================================================================
def obter_ids_verdadeiros(driver, nome_base):
    """
    Procura todos os compartimentos de um metabolito no Neo4j.
    Usa um truque em Cypher: split() corta a string no ' [' do compartimento.
    Assim 'keratan sulfate I [Cytosol]' vira 'keratan sulfate I'.
    """
    query = """
    MATCH (m:ModelMetabolite)
    // O split corta a string antes de abrir o parêntesis reto do compartimento
    WHERE split(m.name, ' [')[0] = $nome_base
    RETURN m.id AS real_id, m.name AS full_name
    """
    
    ids_encontrados = []
    with driver.session() as session:
        result = session.run(query, nome_base=nome_base)
        for record in result:
            ids_encontrados.append(record["real_id"])
            print(f"🔍 Encontrado no Neo4j: {record['full_name']} -> ID: {record['real_id']}")
            
    return ids_encontrados

# =================================================================
# PASSO 2: A tua Função de Injeção Corrigida (usando m.id)
# =================================================================
def inject_pubmed_article(driver, entity_id, pub):
    query = """
    MERGE (p:Publication {id: $pmid_curie})
    ON CREATE SET 
        p.title = $title,
        p.abstract = $abstract,
        p.year = $year,
        p.doi = $doi,
        p.pubmed_url = $url,
        p.biolink_categories = 'Publication'
    
    WITH p
    
    // Procura o ID exato que sacámos do próprio Neo4j no passo 1!
    MATCH (entity:ModelMetabolite)
    WHERE entity.id = $entity_id
    
    MERGE (entity)-[:MENTIONED_IN_PUBLICATION {source: 'pubmed_api'}]->(p)
    """
    
    params = {
        "pmid_curie": f"PMID:{pub['pmid']}",
        "title": pub["title"],
        "abstract": pub["abstract"],
        "year": int(pub["year"]) if pub["year"] else None,
        "doi": pub.get("doi", ""),
        "url": pub.get("pubmed_url", ""),
        "entity_id": entity_id
    }
    
    with driver.session() as session:
        session.run(query, params)


# =================================================================
# O TEU WORKFLOW PRINCIPAL
# =================================================================

query_biologica = "keratan sulfate I"

# 1. Pergunta ao Neo4j: "Que IDs tens para este gajo em qualquer compartimento?"
ids_reais = obter_ids_verdadeiros(driver, query_biologica)

if not ids_reais:
    print(f"❌ Não encontrei '{query_biologica}' no Neo4j. Abortar PubMed.")
else:
    print(f"✅ Vou ligar os artigos a estes {len(ids_reais)} IDs!")
    
    # 2. Fazes a tua pesquisa e o Parse do XML do PubMed AQUI (código da mensagem anterior)
    # publications_data = [...] 
    
    # 3. O Loop Final que faz a magia!
    for pub in publications_data:
        for real_id in ids_reais:
            inject_pubmed_article(driver, real_id, pub)
            print(f"🔗 Artigo {pub['pmid']} ligado ao nó {real_id}!")

driver.close()

🔍 Encontrado no Neo4j: keratan sulfate I [Golgi apparatus] -> ID: m02278g[g]
🔍 Encontrado no Neo4j: keratan sulfate I [Lysosome] -> ID: m02278l[l]
🔍 Encontrado no Neo4j: keratan sulfate I [Extracellular] -> ID: m02278s[s]
✅ Vou ligar os artigos a estes 3 IDs!
🔗 Artigo 41616943 ligado ao nó m02278g[g]!
🔗 Artigo 41616943 ligado ao nó m02278l[l]!
🔗 Artigo 41616943 ligado ao nó m02278s[s]!
🔗 Artigo 41614872 ligado ao nó m02278g[g]!
🔗 Artigo 41614872 ligado ao nó m02278l[l]!
🔗 Artigo 41614872 ligado ao nó m02278s[s]!
🔗 Artigo 41513091 ligado ao nó m02278g[g]!
🔗 Artigo 41513091 ligado ao nó m02278l[l]!
🔗 Artigo 41513091 ligado ao nó m02278s[s]!
🔗 Artigo 41499318 ligado ao nó m02278g[g]!
🔗 Artigo 41499318 ligado ao nó m02278l[l]!
🔗 Artigo 41499318 ligado ao nó m02278s[s]!
🔗 Artigo 41475749 ligado ao nó m02278g[g]!
🔗 Artigo 41475749 ligado ao nó m02278l[l]!
🔗 Artigo 41475749 ligado ao nó m02278s[s]!
